# Generate well-level bulk profiles profiles

NOTE: We are normalizing the bulk-profile plates to the negative controls as the "standard".

## Import libraries

In [1]:
import os
import pathlib
import pprint

import pandas as pd

from pycytominer import aggregate, annotate, normalize, feature_select

## Set paths and variables

In [2]:
# get the batch to process from environment variable
batch_to_process = os.environ.get("BATCH", "batch_1")
if batch_to_process is None:
    raise ValueError(
        "Please set the BATCH environment variable before running this script."
    )

# base directory where batches are located
base_dir = pathlib.Path("./data/").resolve(strict=True)

# Decide what to process
if batch_to_process:
    print(f"Processing {batch_to_process}")
    batch_dirs = [base_dir / batch_to_process]
else:
    print("No specific batch set, processing all available batches")
    batch_dirs = [p for p in base_dir.glob("batch_*") if p.is_dir()]

# path for platemap directory
platemap_dir = pathlib.Path("../metadata/updated_platemaps/")

# Load the barcode_platemap file
barcode_platemap_df = pd.read_csv(
    (platemap_dir / "updated_barcode_platemap.csv").resolve()
)

# operations to perform for feature selection
feature_select_ops = [
    "variance_threshold",
    "correlation_threshold",
    "blocklist",
    "drop_na_columns",
]

Processing batch_1


## Set dictionary with plates to process

In [3]:
plate_info_dictionary = {}

# Loop over batches and layouts
for batch_dir in batch_dirs:
    layouts = [p for p in batch_dir.iterdir() if p.is_dir()]  # all layouts
    for layout_dir in layouts:
        qc_labeled_dir = layout_dir / "qc_labeled_profiles"
        output_dir = layout_dir / "bulk_profiles"
        output_spherize_dir = layout_dir / "spherized_bulk_profiles"

        # Create directories once per layout
        for d in [output_dir, output_spherize_dir]:
            d.mkdir(parents=True, exist_ok=True)

        # Extract plate names from parquet files
        parquet_files = list(qc_labeled_dir.glob("*.parquet"))
        plate_names = [
            "_".join(f.stem.split("_")[:2]) if len(f.stem.split("_")) >= 2 else f.stem
            for f in parquet_files
        ]

        for name in plate_names:
            # Find the corresponding parquet file
            matching_files = [f for f in parquet_files if name in f.stem]
            if not matching_files:
                continue
            profile_path = matching_files[0].resolve(strict=True)

            # Find corresponding platemap CSV
            platemap_row = barcode_platemap_df.loc[
                barcode_platemap_df["plate_barcode"] == name
            ]
            if platemap_row.empty:
                raise ValueError(f"No platemap found for plate {name}")
            platemap_path = (
                platemap_dir / f"{platemap_row['platemap_file'].values[0]}.csv"
            ).resolve(strict=True)

            # Add to dictionary
            plate_info_dictionary[name] = {
                "profile_path": profile_path,
                "platemap_path": platemap_path,
                "output_dir": output_dir,
                "spherize_output_dir": output_spherize_dir,
            }

# View dictionary
print("Number of plates to process:", len(plate_info_dictionary))
pprint.pprint(plate_info_dictionary, indent=4)

Number of plates to process: 16
{   'CARD-CelIns-CX7_251023210001': {   'output_dir': PosixPath('/home/erikserrano/Projects/targeted_fibrosis_drug_screen/3.preprocessing_features/data/batch_1/platemap_1/bulk_profiles'),
                                        'platemap_path': PosixPath('/home/erikserrano/Projects/targeted_fibrosis_drug_screen/metadata/updated_platemaps/Target_Selective_Library_Screen_Plate_1_with_pathways.csv'),
                                        'profile_path': PosixPath('/home/erikserrano/Data-repo/CFReT_screen_data/screen_profiles/batch_1/platemap_1/qc_labeled_profiles/CARD-CelIns-CX7_251023210001_qc_labeled.parquet'),
                                        'spherize_output_dir': PosixPath('/home/erikserrano/Projects/targeted_fibrosis_drug_screen/3.preprocessing_features/data/batch_1/platemap_1/spherized_bulk_profiles')},
    'CARD-CelIns-CX7_251124150001': {   'output_dir': PosixPath('/home/erikserrano/Projects/targeted_fibrosis_drug_screen/3.preprocessing_fe

## Process data with pycytominer


In [4]:
for plate, info in plate_info_dictionary.items():
    output_dir = info["output_dir"]
    print("Performing preprocessing on", plate, output_dir)

    # Use the output_dir from the dictionary for this specific plate
    output_dir = info["output_dir"]

    # generating all output file paths using the output_dir from the dictionary
    output_annotated_file = str(output_dir / f"{plate}_bulk_annotated.parquet")
    output_normalized_file = str(output_dir / f"{plate}_bulk_normalized.parquet")
    output_feature_select_file = str(
        output_dir / f"{plate}_bulk_feature_selected.parquet"
    )

    # loading profiles
    profile_df = pd.read_parquet(info["profile_path"])
    platemap_df = pd.read_csv(info["platemap_path"])

    # Drop all rows in the profiles that failed any Metadata_cqc columns
    cqc_columns = [col for col in profile_df.columns if col.startswith("Metadata_cqc")]
    if cqc_columns:
        profile_df = profile_df[~profile_df[cqc_columns].any(axis=1)]

    # Step 1: Aggregate single-cell data to the well-level using the median
    print("Performing aggregation for", plate, "...")
    aggregated_df = aggregate(
        population_df=profile_df,
        operation="median",
        strata=["Image_Metadata_Plate", "Image_Metadata_Well"],
    )

    # Step 2: Annotation
    print("Performing annotation for", plate, "...")
    annotate(
        profiles=aggregated_df,
        platemap=platemap_df,
        join_on=["Metadata_well_position", "Image_Metadata_Well"],
        output_type="parquet",
        output_file=output_annotated_file,
    )

    # Load the annotated parquet file to fix metadata columns names
    annotated_df = pd.read_parquet(output_annotated_file)

    # Rename columns
    annotated_df.rename(columns={"Image_Metadata_Site": "Metadata_Site"}, inplace=True)

    # Save annotated profiles back to parquet
    annotated_df.to_parquet(output_annotated_file, index=False)

    # Step 3: Normalization (mad robustize)
    # Per plate normalization using the negative controls as the reference population
    print("Performing normalization for", plate, "...")
    neg_control_query = "Metadata_treatment == 'DMSO' and Metadata_cell_type == 'failing'"
    normalize(
        profiles=annotated_df,
        method="mad_robustize",
        samples=neg_control_query,
        output_type="parquet",
        output_file=output_normalized_file,
    )

    # Step 4: Per plate feature selection
    print("Performing feature selection for", plate, "...")
    feature_select(
        profiles=output_normalized_file,
        operation=feature_select_ops,
        na_cutoff=0,
        blocklist_file="./blocklist_features.txt",
        corr_threshold=0.95,
        freq_cut=0.05,
        output_type="parquet",
        output_file=output_feature_select_file,
    )

    print(
        f"Aggregation, annotation, normalization, and feature selection complete for {plate}"
    )

Performing preprocessing on CARD-CelIns-CX7_251213150001 /home/erikserrano/Projects/targeted_fibrosis_drug_screen/3.preprocessing_features/data/batch_1/platemap_4/bulk_profiles
Performing aggregation for CARD-CelIns-CX7_251213150001 ...
Performing annotation for CARD-CelIns-CX7_251213150001 ...
Performing normalization for CARD-CelIns-CX7_251213150001 ...
Performing feature selection for CARD-CelIns-CX7_251213150001 ...
Aggregation, annotation, normalization, and feature selection complete for CARD-CelIns-CX7_251213150001
Performing preprocessing on CARD-CelIns-CX7_251212180001 /home/erikserrano/Projects/targeted_fibrosis_drug_screen/3.preprocessing_features/data/batch_1/platemap_4/bulk_profiles
Performing aggregation for CARD-CelIns-CX7_251212180001 ...
Performing annotation for CARD-CelIns-CX7_251212180001 ...
Performing normalization for CARD-CelIns-CX7_251212180001 ...
Performing feature selection for CARD-CelIns-CX7_251212180001 ...
Aggregation, annotation, normalization, and feat

## Sphering profiles

We concatenate all normalized replicate plates of the same platemap and perform a two-layer feature selection procedure.

In the first layer, we apply feature selection to the concatenated normalized profiles.
In the second layer, we remove low variance features in the concatenated normalized profiles _for the DMSO control wells only_.

We then apply a sphering transform using this concatenated feature selected data.
We use the negative-control wells as the reference population to decorrelate the features to place profiles on a shared control-based covariance scale.

### Parameters:

- `neg_control_query`: pandas query string that selects the control wells used to fit both standardization and sphering. Here, the reference population is failing-cell DMSO wells.

In [5]:
# setting negative control query 
neg_control_query = "Metadata_treatment == 'DMSO' and Metadata_cell_type == 'failing'"

# Group annotated replicate profiles by plate map for pooled normalization and 
# sphering outputs. 
# {"platemap_1": {"replicate_paths": [path1, path2], "spherized_output_dir": path}, ...}
normalized_replicate_plates = {}

for plate_barcode, info in plate_info_dictionary.items():
    # Example key: platemap_1, platemap_2, etc.
    plate_key = info["output_dir"].parent.name

    # Path to the well-level normalized profile generated in the previous step.
    normalized_plate_path = (
        info["output_dir"] / f"{plate_barcode}_bulk_normalized.parquet"
    )

    # Start a new group the first time we see this plate map.
    if plate_key not in normalized_replicate_plates:
        normalized_replicate_plates[plate_key] = {
            "replicate_paths": [],
            "spherized_output_dir": info["spherize_output_dir"],
        }

    # Add this physical replicate plate to its plate-map group.
    normalized_replicate_plates[plate_key]["replicate_paths"].append(
        normalized_plate_path
    )

# sort based on plate key
normalized_replicate_plates = dict(sorted(normalized_replicate_plates.items()))


In [6]:
# Sphering step:
# Process each plate map after pooling its replicate plates.
for plate_key, plate_info in normalized_replicate_plates.items():
    replicate_paths = sorted(plate_info["replicate_paths"])
    spherized_output_dir = plate_info["spherized_output_dir"]

    # All replicate paths in this group share the same bulk output directory.
    output_dir = replicate_paths[0].parent
    spherized_output_dir.mkdir(parents=True, exist_ok=True)

    # Platemap-level outputs created from pooled replicate plates.
    output_feature_select_file = (
        output_dir / f"{plate_key}_replicate_bulk_feature_selected.parquet"
    )
    output_spherized_file = (
        spherized_output_dir / f"{plate_key}_replicate_bulk_spherized.parquet"
    )

    # step 1: concat mad-normalized replicate plates before feature selection
    print(f"Concatenating replicate plates for {plate_key}...")
    concat_replicate_df = pd.concat(
        [pd.read_parquet(path) for path in replicate_paths],
        ignore_index=True,
    ).reset_index(drop=True)

    # step 2a: Apply feature selection on the pooled replicate plates to get a common 
    # set of features for sphering.
    print(f"Feature selecting {plate_key}...")
    feature_select_df = feature_select(
        profiles=concat_replicate_df,
        operation=feature_select_ops,
        na_cutoff=0,
        blocklist_file="./blocklist_features.txt",
        corr_threshold=0.95,
        freq_cut=0.05
    )

    # step 2b: Remove features with too little variation inside the exact control
    # population used to fit spherization.
    print(f"Feature selecting {plate_key} with variance threshold "
          "within negative controls only...")
    zero_negcon_var_fs_df = feature_select(
        profiles=output_feature_select_file,
        operation="variance_threshold",
        freq_cut=0.05,
        unique_cut=0.01,
        samples=neg_control_query,
    )

    # step 3: Spherize/whiten all profiles using the pooled negative controls as the
    # reference population.
    print(f"Sphering {plate_key} using pooled negative controls...")
    normalize(
        profiles=zero_negcon_var_fs_df,
        method="spherize",
        samples=neg_control_query,
        spherize_center=True,
        spherize_method="ZCA-cor",
        spherize_epsilon=1e-6,
        output_file=output_spherized_file,
        output_type="parquet",
    )

    print(f"Saved feature-selected profiles to {output_feature_select_file}")
    print(f"Saved spherized profiles to {output_spherized_file}")


Concatenating replicate plates for platemap_1...
Feature selecting platemap_1...
Feature selecting platemap_1 with variance threshold within negative controls only...
Sphering platemap_1 using pooled negative controls...
Saved feature-selected profiles to /home/erikserrano/Projects/targeted_fibrosis_drug_screen/3.preprocessing_features/data/batch_1/platemap_1/bulk_profiles/platemap_1_replicate_bulk_feature_selected.parquet
Saved spherized profiles to /home/erikserrano/Projects/targeted_fibrosis_drug_screen/3.preprocessing_features/data/batch_1/platemap_1/spherized_bulk_profiles/platemap_1_replicate_bulk_spherized.parquet
Concatenating replicate plates for platemap_2...
Feature selecting platemap_2...
Feature selecting platemap_2 with variance threshold within negative controls only...
Sphering platemap_2 using pooled negative controls...
Saved feature-selected profiles to /home/erikserrano/Projects/targeted_fibrosis_drug_screen/3.preprocessing_features/data/batch_1/platemap_2/bulk_prof

In [7]:
# Check an example output file
test_df = pd.read_parquet(output_spherized_file)

print(test_df.shape)
print("Plate:", test_df.Metadata_Plate.unique())
print(
    "Metadata columns:", [col for col in test_df.columns if col.startswith("Metadata_")]
)
test_df.head(2)

(216, 913)
Plate: ['CARD-CelIns-CX7_251211180001' 'CARD-CelIns-CX7_251212100001'
 'CARD-CelIns-CX7_251212180001' 'CARD-CelIns-CX7_251213150001']
Metadata columns: ['Metadata_WellRow', 'Metadata_WellCol', 'Metadata_heart_number', 'Metadata_cell_type', 'Metadata_heart_failure_type', 'Metadata_treatment', 'Metadata_Pathway', 'Metadata_Plate', 'Metadata_Well']


,Metadata_WellRow,Metadata_WellCol,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Pathway,Metadata_Plate,Metadata_Well,Cytoplasm_AreaShape_Area,...,Nuclei_Texture_InfoMeas2_DNA_3_01_256,Nuclei_Texture_InfoMeas2_DNA_3_02_256,Nuclei_Texture_InfoMeas2_ER_3_02_256,Nuclei_Texture_InfoMeas2_Mito_3_02_256,Nuclei_Texture_InfoMeas2_PM_3_02_256,Nuclei_Texture_InverseDifferenceMoment_Actin_3_02_256,Nuclei_Texture_InverseDifferenceMoment_ER_3_00_256,Nuclei_Texture_InverseDifferenceMoment_PM_3_00_256,Nuclei_Texture_Variance_ER_3_00_256,Nuclei_Texture_Variance_Mito_3_01_256
0,B,2,7,healthy,None,DMSO,None,CARD-CelIns-CX7_251211180001,B02,-0.242142,...,0.222049,0.233864,0.509243,0.471846,0.471808,0.722417,-0.330117,-0.194789,0.185246,-0.039242
1,B,3,25,failing,dilated_cardiomyopathy,UCD-0001078,Metabolism,CARD-CelIns-CX7_251211180001,B03,-0.115787,...,-0.179263,-0.225489,0.490450,0.739377,0.151224,1.308337,-0.546861,-0.770529,-0.864865,2.524586
